# AMDockVS Vina Virtual Screening: 100-Ligand Smoke Test

This notebook validates a manual virtual-screening workflow with the current runtime:

1. Import ligands through MolSuite jobs.
2. Filter by n_atoms > 10, 150 <= MW <= 500, and a single molecular fragment.
3. Compute descriptors.
4. Prepare ligands.
5. Import and prepare the receptor.
6. Read the box from a file.
7. Run Vina on 100 ligands with vina_cpu=1.

The notebook does not create an automated workflow; it composes explicit runtime calls.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path

REPO_ROOT = Path("/home/mario/PycharmProjects/AMDockVS")
MOLSUITE_ROOT = Path("/")

for path in (REPO_ROOT / "src", MOLSUITE_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from amdockvs import AMDockVSRuntime

DATA_DIR = REPO_ROOT / "data"
LIGAND_FILE = DATA_DIR / "Enamine_D10k" / "Enamine_D10k.sdf"
RECEPTOR_FILE = DATA_DIR / "docking" / "1iep_receptorH.pdb"
BOX_FILE = DATA_DIR / "docking" / "1iep_receptor.box.txt"

PROJECT_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PROJECT_DIR = REPO_ROOT / "notebooks" / "_tmp" / f"09_vina_vs_100_ligands_smoke_project_{PROJECT_RUN_ID}"
PROJECT_NAME = f"09_vina_vs_100_ligands_smoke_{PROJECT_RUN_ID}"

# Import more than 100 because the atom-count/MW/fragment filters can discard entries.
MAX_IMPORT_RECORDS = 1000
VS_LIGAND_COUNT = 100

# 100 ligands, Vina with 1 CPU per task. batch_size=1 allows several chunks to run in parallel.
DOCKING_BATCH_SIZE = 1
VINA_CPU = 1
VINA_EXHAUSTIVENESS = 1
VINA_NUM_MODES = 3

# Set to False to prepare the project without running Vina.
RUN_DOCKING = True

for required in (LIGAND_FILE, RECEPTOR_FILE, BOX_FILE):
    if not required.exists():
        raise FileNotFoundError(required)

print({
    "ligands": str(LIGAND_FILE),
    "receptor": str(RECEPTOR_FILE),
    "box": str(BOX_FILE),
    "project": str(PROJECT_DIR),
})

{'ligands': '/home/mario/PycharmProjects/AMDockVS/data/Enamine_D10k/Enamine_D10k.sdf', 'receptor': '/home/mario/PycharmProjects/AMDockVS/data/docking/1iep_receptorH.pdb', 'box': '/home/mario/PycharmProjects/AMDockVS/data/docking/1iep_receptor.box.txt', 'project': '/home/mario/PycharmProjects/AMDockVS/notebooks/_tmp/09_vina_vs_100_ligands_smoke_project_20260514_173852'}


## Local utilities

Helpers for waiting on jobs and parsing the box. Ligand and receptor filtering is delegated to the runtime, which creates refs through jobs.

In [2]:
def require_completed(stage: str, status_map):
    failed = {job_id: row.status for job_id, row in status_map.items() if row.status != "completed"}
    if failed:
        raise RuntimeError(f"{stage} failed: {failed}")
    return status_map


def parse_vina_box(path: Path) -> tuple[tuple[float, float, float], tuple[float, float, float]]:
    values: dict[str, float] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, value = [part.strip() for part in line.split("=", 1)]
        values[key] = float(value)
    center = (values["center_x"], values["center_y"], values["center_z"])
    size = (values["size_x"], values["size_y"], values["size_z"])
    return center, size


box_center, box_size = parse_vina_box(BOX_FILE)
box_center, box_size

((15.19, 53.903, 16.917), (20.0, 20.0, 20.0))

## Create a project and load ligands

Import is delegated to MolSuite through runtime.loader.load_ligands(...), which returns job_id.

In [3]:
runtime = AMDockVSRuntime()
runtime.create_project(PROJECT_NAME, PROJECT_DIR)

ligand_import_jobs = runtime.loader.load_ligands(
    [LIGAND_FILE],
    max_records=MAX_IMPORT_RECORDS,
    batch_size=250,
    executor_name="process",
)
ligand_import_status = require_completed(
    "ligand import",
    runtime.wait_for_jobs(ligand_import_jobs, timeout_s=900.0, poll_s=0.5),
)
ligand_import_status

[2026-05-14 17:38:52,504] [  INFO  ] amdockvs.app.lifecycle: Logging manager started
[2026-05-14 17:38:52,566] [  INFO  ] amdockvs.executor.core: Registered thread executor 'thread' (max_workers=32)
[2026-05-14 17:38:52,569] [  INFO  ] amdockvs.executor.core: Registered process executor 'process' (dynamic CPU allocation)
[2026-05-14 17:38:52,635] [  INFO  ] amdockvs.executor.core: Registered process pool executor 'process_pool_forkserver' (backend=stdlib max_workers=16 start_method=forkserver max_tasks_per_child=50)
[2026-05-14 17:38:52,638] [  INFO  ] amdockvs.executor.core: Executor manager started (total_cpu=16)
[2026-05-14 17:38:52,638] [  INFO  ] amdockvs.app.core: MolSuite runtime ready: executor=/home/mario/PycharmProjects/AMDockVS/notebooks/_tmp/09_vina_vs_100_ligands_smoke_project_20260514_173852/executor.db
[2026-05-14 17:38:53,023] [  INFO  ] amdockvs.project.core: Project context active: id=ee8df57c-5aaf-4dac-95d4-402d4b9d28d3 path=/home/mario/PycharmProjects/AMDockVS/noteb

{'4346ad95696a48648220ff76d9810a9b': JobStatus(job_id='4346ad95696a48648220ff76d9810a9b', project_id='ee8df57c-5aaf-4dac-95d4-402d4b9d28d3', origin_id='amdockvs', task_type='amdock_load_ligands_file_job', status='completed', executor_name='process', progress=100.0, progress_structural=100.0, progress_operational=100.0, progress_running_chunks_avg=0.0, priority=0, queue_policy='fifo', chunks_total=4, chunks_emitted=4, chunks_dispatched=4, chunks_done=4, chunks_failed=0, chunks_stage_failed=0, chunks_running=0, chunks_pending=0, chunks_staging=0, chunks_ready_not_dispatched=0, feed_exhausted=True, output_sink=None, job_queue_wait_s=0.604712, chunks_started=4, chunk_queue_wait_avg_s=0.4722665995359421, chunk_queue_wait_max_s=0.5289852619171143, feed_cursor_position=4, feed_items_acked=4, loop_latency_ms=31.190970999887213, throughput_eps=2.759331, running_cpu=0, max_job_cpu=None, scheduler_block_reason='', last_dispatch_attempt_at=datetime.datetime(2026, 5, 14, 17, 38, 53, 747252), last_s

[2026-05-14 17:38:55,277] [  INFO  ] amdockvs.executor.core: Submitted job f7e1bdf802ba4538842bf2ed247f05d5 executor=process runner=amdock.qsar.jobs:calculate_descriptor_batch inflight=16 priority=0 [project_id=ee8df57c-5aaf-4dac-95d4-402d4b9d28d3 job_id=f7e1bdf802ba4538842bf2ed247f05d5]
[2026-05-14 17:38:56,874] [  INFO  ] amdockvs.executor.core: Submitted job 161db44a22e24f899f9bc5251bb7746e executor=process runner=amdock.chemistry.jobs:ligand_chemistry_task inflight=16 priority=0 [project_id=ee8df57c-5aaf-4dac-95d4-402d4b9d28d3 job_id=161db44a22e24f899f9bc5251bb7746e]
[2026-05-14 17:39:00,469] [  INFO  ] amdockvs.executor.core: Submitted job 2c27cb4d74134caa91f4d9c204df84b8 executor=process runner=amdock.docking.preparation_jobs:_PrepareEntitiesVinaJobSpec.run_chunk inflight=16 priority=0 [project_id=ee8df57c-5aaf-4dac-95d4-402d4b9d28d3 job_id=2c27cb4d74134caa91f4d9c204df84b8]
[2026-05-14 17:39:03,152] [  INFO  ] amdockvs.executor.core: Submitted job b7ae63365e174762bf2b41c29432cedd

## Compute RDKit descriptors

Compute descriptors for the imported ligands. The molecular-weight filter uses the amdock_molecules.mw column, populated by runtime.qsar.compute_descriptors(...).

In [4]:
descriptor_job = runtime.qsar.compute_descriptors(
    molecule_set=runtime.molecules.select(role="ligand"),
    batch_size=250,
    executor_name="process",
    depends_on=ligand_import_jobs,
)
descriptor_status = runtime.wait_for_job(descriptor_job, poll_s=0.5)
if descriptor_status.status != "completed":
    raise RuntimeError(descriptor_status)

descriptor_status

JobStatus(job_id='f7e1bdf802ba4538842bf2ed247f05d5', project_id='ee8df57c-5aaf-4dac-95d4-402d4b9d28d3', origin_id='amdockvs', task_type='amdock_calculate_molecule_descriptors_job', status='completed', executor_name='process', progress=100.0, progress_structural=100.0, progress_operational=100.0, progress_running_chunks_avg=0.0, priority=0, queue_policy='fifo', chunks_total=4, chunks_emitted=4, chunks_dispatched=4, chunks_done=4, chunks_failed=0, chunks_stage_failed=0, chunks_running=0, chunks_pending=0, chunks_staging=0, chunks_ready_not_dispatched=0, feed_exhausted=True, output_sink=None, job_queue_wait_s=0.082083, chunks_started=4, chunk_queue_wait_avg_s=0.0557631254196167, chunk_queue_wait_max_s=0.07201731204986572, feed_cursor_position=4, feed_items_acked=4, loop_latency_ms=16.08785698772408, throughput_eps=2.915609, running_cpu=0, max_job_cpu=None, scheduler_block_reason='', last_dispatch_attempt_at=datetime.datetime(2026, 5, 14, 17, 38, 55, 363659), last_scheduler_reason_at=None,

## Filter by atom count, molecular weight, and fragments

Criteria:

- n_atoms > 10
- 150 Da <= MW <= 500 Da
- One molecular fragment, because Meeko/Vina expect one ligand per input
- Select 100 ligands for virtual screening

The filter remains a lazy MoleculeScope; the notebook does not open sessions or materialize full ligand lists.

In [5]:
vs_ligand_scope = runtime.molecules.filter(
    runtime.molecules.select(
        role="ligand",
        n_atoms_gt=10,
        mw_min=150.0,
        mw_max=500.0,
        limit=VS_LIGAND_COUNT,
    ),
    filters={"fragment_count": 1},
)

selected_count = runtime.molecules.count(vs_ligand_scope)
if selected_count != VS_LIGAND_COUNT:
    raise RuntimeError(f"Selected {selected_count} ligands; expected {VS_LIGAND_COUNT}.")

{"ligand_scope": vs_ligand_scope, "selected": selected_count}


{'ligand_scope': MoleculeScope(filters={'primary_role': 'ligand', 'n_atoms__gt': 10, 'mw__gte': 150.0, 'mw__lte': 500.0, 'fragment_count': 1}, source_set_id=None, order=('id',), limit=100),
 'selected': 100}

## Generate 3D ligand conformations

Meeko preparation requires 3D ligands. This step generates 3D conformers for the filtered set before preparing PDBQT.

In [6]:
generate_3d_job = runtime.chemistry.generate_3d_ligands(
    ligands=vs_ligand_scope,
    add_hs=True,
    optimize=True,
    batch_size=50,
    executor_name="process",
)
generate_3d_status = runtime.wait_for_job(generate_3d_job, poll_s=0.5)
if generate_3d_status.status != "completed":
    raise RuntimeError(generate_3d_status)

generate_3d_status

RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generated a 3D conformer: 0
RDKit generate

JobStatus(job_id='161db44a22e24f899f9bc5251bb7746e', project_id='ee8df57c-5aaf-4dac-95d4-402d4b9d28d3', origin_id='amdockvs', task_type='amdock_ligand_chemistry_job', status='completed', executor_name='process', progress=100.0, progress_structural=100.0, progress_operational=100.0, progress_running_chunks_avg=0.0, priority=0, queue_policy='fifo', chunks_total=2, chunks_emitted=2, chunks_dispatched=2, chunks_done=2, chunks_failed=0, chunks_stage_failed=0, chunks_running=0, chunks_pending=0, chunks_staging=0, chunks_ready_not_dispatched=0, feed_exhausted=True, output_sink=None, job_queue_wait_s=0.04173, chunks_started=2, chunk_queue_wait_avg_s=0.02649351954460144, chunk_queue_wait_max_s=0.035002827644348145, feed_cursor_position=2, feed_items_acked=2, loop_latency_ms=10.809933010023087, throughput_eps=0.62173, running_cpu=0, max_job_cpu=None, scheduler_block_reason='', last_dispatch_attempt_at=datetime.datetime(2026, 5, 14, 17, 38, 56, 915795), last_scheduler_reason_at=None, last_schedul

## Prepare ligands for Vina

Preparation runs as a job and consumes the filtered MoleculeScope. It expects ligands with generated 3D conformations.

In [7]:
prepare_ligands_job = runtime.docking.prepare_ligands(
    ligand_set=vs_ligand_scope,
    batch_size=50,
    engine="vina",
    force=True,
    executor_name="process",
)
prepare_ligands_status = runtime.wait_for_job(prepare_ligands_job, poll_s=0.5)
if prepare_ligands_status.status != "completed":
    raise RuntimeError(prepare_ligands_status)

prepare_ligands_status

JobStatus(job_id='2c27cb4d74134caa91f4d9c204df84b8', project_id='ee8df57c-5aaf-4dac-95d4-402d4b9d28d3', origin_id='amdockvs', task_type='amdock_prepare_ligands_job', status='completed', executor_name='process', progress=100.0, progress_structural=100.0, progress_operational=100.0, progress_running_chunks_avg=0.0, priority=0, queue_policy='fifo', chunks_total=2, chunks_emitted=2, chunks_dispatched=2, chunks_done=2, chunks_failed=0, chunks_stage_failed=0, chunks_running=0, chunks_pending=0, chunks_staging=0, chunks_ready_not_dispatched=0, feed_exhausted=True, output_sink=None, job_queue_wait_s=0.078563, chunks_started=2, chunk_queue_wait_avg_s=0.0674910843372345, chunk_queue_wait_max_s=0.07197707891464233, feed_cursor_position=2, feed_items_acked=2, loop_latency_ms=10.498581017600372, throughput_eps=0.79364, running_cpu=0, max_job_cpu=None, scheduler_block_reason='', last_dispatch_attempt_at=datetime.datetime(2026, 5, 14, 17, 39, 0, 558820), last_scheduler_reason_at=None, last_scheduler_

## Load and prepare the receptor

Import /home/mario/PycharmProjects/AMDockVS/data/docking/1iep_receptorH.pdb through a job, then prepare it for Vina.

In [8]:
receptor_import_jobs = runtime.loader.load_receptors(
    [RECEPTOR_FILE],
    batch_size=1,
    executor_name="process",
)
receptor_import_status = require_completed(
    "receptor import",
    runtime.wait_for_jobs(receptor_import_jobs, timeout_s=300.0, poll_s=0.5),
)

receptor_scope = runtime.molecules.select(role="receptor", limit=1)
receptor = next(runtime.molecules.stream(receptor_scope))
receptor_id = int(receptor.id or 0)

prepare_receptor_job = runtime.docking.prepare_receptors(
    receptor_set=receptor_scope,
    batch_size=1,
    engine="vina",
    force=True,
    executor_name="process",
)
prepare_receptor_status = runtime.wait_for_job(prepare_receptor_job, poll_s=0.5)
if prepare_receptor_status.status != "completed":
    raise RuntimeError(prepare_receptor_status)

{"receptor_id": receptor_id, "receptor_scope": receptor_scope, "status": prepare_receptor_status.status}


@> 4412 atoms and 1 coordinate set(s) were parsed in 0.02s.


{'receptor_id': 1001,
 'receptor_scope': MoleculeScope(filters={'primary_role': 'receptor'}, source_set_id=None, order=('id',), limit=1),
 'status': 'completed'}

## Define the docking box

Read the box from 1iep_receptor.box.txt and save it in the receptor metadata with docking.set_grid(...).

In [9]:
grid = runtime.docking.set_grid(
    receptor_id=receptor_id,
    center=box_center,
    size=box_size,
    spacing=0.375,
    engine="vina",
    metadata={"source": str(BOX_FILE)},
)

grid

{'receptor_id': 1001,
 'engine': 'vina',
 'center': [15.19, 53.903, 16.917],
 'size': [20.0, 20.0, 20.0],
 'spacing': 0.375}

## Check Vina requirements

Before submitting the docking job, verify that the ligands and receptor have prepared_vina=True and that the receptor has grid_vina=True.

In [10]:
requirement_check = runtime.docking.check_required(
    method="vina",
    ligand_set=vs_ligand_scope,
    receptor_set=receptor_scope,
)
if not requirement_check["ready"]:
    raise RuntimeError(requirement_check)

requirement_check


{'ready': True,
 'operation': 'docking.vina',
 'missing': {'ligands_prepared_for_vina': [],
  'receptors_prepared_for_vina': [],
  'receptor_grids_vina': []},
 'counts': {'ligands_total': 100,
  'ligands_ready': 100,
  'receptors_total': 1,
  'receptors_ready': 1,
  'receptor_grids_ready': 1}}

## Run virtual screening with Vina

Run on 100 ligands and one receptor with vina_cpu=1. The notebook uses batch_size=1 to create independent chunks; MolSuite can dispatch multiple Vina tasks in parallel, each using one CPU.

In [11]:
if RUN_DOCKING:
    docking_job = runtime.docking.run(
        ligand_set=vs_ligand_scope,
        receptor_set=receptor_scope,
        batch_size=DOCKING_BATCH_SIZE,
        exhaustiveness=VINA_EXHAUSTIVENESS,
        num_modes=VINA_NUM_MODES,
        box_center=box_center,
        box_size=box_size,
        scoring_function="vina",
        vina_cpu=VINA_CPU,
        seed=42,
        spacing=0.375,
        energy_range=3.0,
        min_rmsd=1.0,
        executor_name="process",
    )
    docking_status = runtime.wait_for_job(docking_job, poll_s=1.0)
    if docking_status.status != "completed":
        raise RuntimeError(docking_status)
else:
    docking_job = None
    docking_status = None

{"docking_job": docking_job, "status": None if docking_status is None else docking_status.status}


{'docking_job': 'd948cbc272f6452ea45a9d2545d813d8', 'status': 'completed'}

## Review results

In [12]:
if RUN_DOCKING:
    top_hits = runtime.docking.top_hits(limit=20, receptor_id=receptor_id)
    print(f"Top hits: {len(top_hits)}")
    top_hits[:5]
else:
    print("RUN_DOCKING=False; no docking results were generated.")

Top hits: 20


## Shut down the runtime

Run this cell when finished to shut down executors and connections.

In [13]:
runtime.shutdown()

[2026-05-14 17:40:29,594] [  INFO  ] amdockvs.app.core: Project closed: id=ee8df57c-5aaf-4dac-95d4-402d4b9d28d3 name=09_vina_vs_100_ligands_smoke_20260514_173852 [project_id=ee8df57c-5aaf-4dac-95d4-402d4b9d28d3]
[2026-05-14 17:40:29,599] [  INFO  ] amdockvs.executor.core: Executor manager stopped
[2026-05-14 17:40:29,600] [  INFO  ] amdockvs.app.lifecycle: Logging manager stopping
